In [ ]:
!pip install wandb evaluate -q

In [ ]:
from huggingface_hub import notebook_login
import wandb
notebook_login()

In [ ]:
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

raw_dataset = load_dataset("wikimedia/wikipedia", "20231101.vi", split="train[:5000]")
split_datasets = raw_dataset.train_test_split(test_size=0.1, seed=42)

In [ ]:
import os
import torch
import wandb
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

In [ ]:
split_datasets["validation"] = split_datasets.pop("test")
split_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'text'],
        num_rows: 4500
    })
    validation: Dataset({
        features: ['id', 'url', 'title', 'text'],
        num_rows: 500
    })
})

In [ ]:
old_tokenizer = AutoTokenizer.from_pretrained("gpt2")

def get_training_corpus():
    for i in range(0, len(split_datasets["train"]), 1000):
        yield split_datasets["train"][i : i + 1000]["text"]

vocab_size = 40000
new_tokenizer = old_tokenizer.train_new_from_iterator(get_training_corpus(), vocab_size=vocab_size)
new_tokenizer.pad_token = new_tokenizer.eos_token

In [ ]:
model = GPT2LMHeadModel.from_pretrained("gpt2")
print(f"-> Kích thước từ điển cũ: {model.config.vocab_size}")
model.resize_token_embeddings(len(new_tokenizer))
print(f"-> Kích thước từ điển mới: {len(new_tokenizer)}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-> Kích thước từ điển cũ: 50257
-> Kích thước từ điển mới: 40000


In [ ]:
def tokenize_function(examples):
    return new_tokenizer(examples["text"])

tokenized_datasets = split_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=["id", "url", "title", "text"]
)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1594 > 1024). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
block_size = 128
def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    return result

lm_datasets = tokenized_datasets.map(group_texts, batched=True)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
os.environ["WANDB_PROJECT"] = "gpt2-vietnamese-training"
repo_name = "gpt2-vietnamese-mini"

data_collator = DataCollatorForLanguageModeling(tokenizer=new_tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=repo_name,
    learning_rate=3e-4,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    num_train_epochs=2,
    fp16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    report_to="wandb",
    push_to_hub=True,
    hub_model_id=f"lilith-aeva/{repo_name}",
    hub_strategy="every_save",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    data_collator=data_collator,
    processing_class=new_tokenizer
)

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 0, 'bos_token_id': 0, 'pad_token_id': 0}.


Epoch,Training Loss,Validation Loss
1,4.056929,4.024884
2,3.775048,3.872960


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=11544, training_loss=4.221794953217378, metrics={'train_runtime': 2693.1353, 'train_samples_per_second': 68.575, 'train_steps_per_second': 4.286, 'total_flos': 1.2063983763456e+16, 'train_loss': 4.221794953217378, 'epoch': 2.0})

In [ ]:
trainer.push_to_hub("Hoàn thành huấn luyện GPT-2 Tiếng Việt Mini")
wandb.finish()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...se-mini/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...se-mini/model.safetensors:  33%|###2      |  152MB /  466MB            

eval/loss,█▁
eval/runtime,▁█
eval/samples_per_second,█▁
eval/steps_per_second,█▁
train/epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇█
train/grad_norm,█▂▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▄█▄████▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
train/loss,█▇▆▆▅▇▆▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,3.87296
eval/runtime,49.021


In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="lilith-aeva/gpt2-vietnamese-mini",
    device=-1
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
split_datasets["train"][1]

{'id': '3753',
 'url': 'https://vi.wikipedia.org/wiki/Ph%E1%BB%A5c%20Sinh',
 'title': 'Phục Sinh',
 'text': 'Phục sinh nghĩa là sự sống lại nói chung của người hoặc các vị thần. \n\nNgoài ra trong tiếng Việt nó có thể là:\n\n Lễ Phục Sinh, liên quan đến Sự phục sinh của Giêsu\n Đảo Phục Sinh\n Trứng Phục Sinh và Thỏ Phục Sinh\n Mùa Phục Sinh\n Cây cọ Phục Sinh\n Chuột phục sinh\n Khác\n Phục Sinh (伏生, 260 TCN?-170 TCN?), một học giả uyên thâm trong nghiên cứu và giảng giải kinh Thượng Thư, người đã đọc cho chép lại 28 (29) thiên của Kinh Thư, gọi là bản kim văn của Kinh Thư.\n Phục sinh (phim) : Phim năm 1960 của Nga\n Phục sinh (tiểu thuyết), tiểu thuyết của Lev Tolstoy.'}

In [ ]:
prompt = "Đảo Phục Sinh"

results = generator(
    prompt,
    max_new_tokens=100,
    num_return_sequences=1,
    do_sample=False,
    temperature=1,
    top_k=50,
    repetition_penalty=1.2
)

print(results[0]["generated_text"])

Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Đảo Phục Sinh, một trong những thành phố lớn nhất thế giới.

Địa lý

Vị trí địa lý
Thành phố nằm ở phía đông nam của tỉnh bang Baden-Württemberg, cách trung tâm thủ đô Hà Nội khoảng 2 km về phía tây bắc, cách Thành phố Hồ Chí Minh khoảng 1 km về phía đông và cách Đà Nẵng khoảng 1 km về phía nam. Địa hình núi cao bao gồm các dãy núi đá vôi thuộc khu vực Đông Nam Á, có độ dốc thấp hơn
